# SVD Landscape Fine-tuning — Local GPU (VS Code)

> Notebook adapté pour tourner **en local** avec VS Code + Jupyter extension.  
> Toutes les dépendances Google Colab / Drive ont été supprimées.

**Prérequis :**
- Python 3.10+ avec un venv ou conda actif
- CUDA installé et `nvidia-smi` fonctionnel
- ~40 Go de disque libre (données + checkpoints)
- Tes scripts `make_dataset_svd.py` et `train_svd_lora.py` dans le même dossier que ce notebook

**Structure des dossiers créée automatiquement :**
```
./svd_project/
  data/          ← clips extraits
  checkpoints/   ← poids LoRA sauvegardés
```

In [1]:
# ── 1. Vérifier le GPU ──────────────────────────────────────────────────────
import subprocess, sys

# result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
# print(result.stdout if result.returncode == 0 else "❌ nvidia-smi introuvable — vérifie ton installation CUDA.")

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")


PyTorch version : 2.2.2
CUDA disponible : False


In [2]:
# ── 2. Installer les dépendances ────────────────────────────────────────────
# Lance cette cellule une seule fois. Recomante le kernel ensuite si nécessaire.
import subprocess, sys

packages = [
    "yt-dlp",
    "opencv-python-headless",
    "diffusers[torch]",
    "transformers",
    "accelerate",
    "peft",
    "imageio[ffmpeg]",
    "tqdm",
    "Pillow",
    "numpy<2",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✅ Dépendances installées.")


✅ Dépendances installées.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# ── 3. Configurer les chemins locaux ────────────────────────────────────────
import os
from pathlib import Path

# ✏️  Modifie ces chemins si tu veux stocker ailleurs
BASE_DIR  = Path("./svd_project")
DATA_DIR  = BASE_DIR / "data"
CKPT_DIR  = BASE_DIR / "checkpoints"

# Chemins vers tes scripts (dans le même dossier que ce notebook par défaut)
NOTEBOOK_DIR    = Path(".").resolve()
MAKE_DATASET_PY = NOTEBOOK_DIR / "make_dataset_svd.py"
TRAIN_PY        = NOTEBOOK_DIR / "train_svd_lora.py"

# Création des dossiers
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Vérification des scripts
for script in [MAKE_DATASET_PY, TRAIN_PY]:
    status = "✅" if script.exists() else "❌ MANQUANT"
    print(f"{status}  {script}")

print(f"\nDonnées   → {DATA_DIR.resolve()}")
print(f"Checkpoints → {CKPT_DIR.resolve()}")


✅  /Users/mat/Desktop/Cours X/3A/MultiModGenAI/Makeitalive/src/SVD/make_dataset_svd.py
✅  /Users/mat/Desktop/Cours X/3A/MultiModGenAI/Makeitalive/src/SVD/train_svd_lora.py

Données   → /Users/mat/Desktop/Cours X/3A/MultiModGenAI/Makeitalive/src/SVD/svd_project/data
Checkpoints → /Users/mat/Desktop/Cours X/3A/MultiModGenAI/Makeitalive/src/SVD/svd_project/checkpoints


In [ ]:
# ── 4. Login HuggingFace (nécessaire pour télécharger les poids SVD) ────────
# Génère ton token sur https://huggingface.co/settings/tokens  (rôle "read")
# Option A : login interactif (recommandé)
from huggingface_hub import login
login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")  # remplace par ton token

# Option B : variable d'environnement (plus sécurisé)
# import os; login(token=os.environ["HF_TOKEN"])

In [5]:
# ── 5. Extraire le dataset ──────────────────────────────────────────────────
import subprocess, sys

# ✏️  Change l'URL et les paramètres selon ton besoin
YOUTUBE_URL = "https://www.youtube.com/watch?v=AKeUssuu3Is"

cmd = [
    sys.executable, str(MAKE_DATASET_PY),
    "--url",       YOUTUBE_URL,
    "--out",       str(DATA_DIR),
    "--clip_len",  "14",
    "--fps",       "7",
    "--size",      "512",
    "--interval",  "4.0",
    "--max_clips", "2000",
]

print("Lancement de l'extraction... (peut prendre plusieurs minutes)")
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("\n✅ Dataset extrait.")


Lancement de l'extraction... (peut prendre plusieurs minutes)
/Users/mat/Desktop/Cours X/3A/MultiModGenAI/Makeitalive/.myenv/bin/python /Users/mat/Desktop/Cours X/3A/MultiModGenAI/Makeitalive/src/SVD/make_dataset_svd.py --url https://www.youtube.com/watch?v=AKeUssuu3Is --out svd_project/data --clip_len 14 --fps 7 --size 512 --interval 4.0 --max_clips 2000
[youtube] Extracting URL: https://www.youtube.com/watch?v=AKeUssuu3Is
[youtube] AKeUssuu3Is: Downloading webpage
[youtube] AKeUssuu3Is: Downloading android vr player API JSON
[youtube] AKeUssuu3Is: Downloading player 74edf1a3-tv
[youtube] [jsc:deno] Solving JS challenges using deno


[info] AKeUssuu3Is: Downloading 1 format(s): 136
[download] Destination: svd_project/data/tmp_landscape.mp4
[download] 100% of    3.64GiB in 00:05:50 at 10.63MiB/s     


Video FPS: 24.0
Frame gap: 25 frames source par frame de clip
Mouvement capturé par clip: 14.6s de vidéo source
Motion filter: [3.0, 40.0]


Clips extracted: 8clips [00:57,  4.77s/clips, motion=23.4, rejected_static=0]Traceback (most recent call last):
  File "/Users/mat/Desktop/Cours X/3A/MultiModGenAI/Makeitalive/src/SVD/make_dataset_svd.py", line 224, in <module>
    extract_clips(
  File "/Users/mat/Desktop/Cours X/3A/MultiModGenAI/Makeitalive/src/SVD/make_dataset_svd.py", line 161, in extract_clips
    if is_scene_cut(prev_raw, next_frame, scene_cut_threshold):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mat/Desktop/Cours X/3A/MultiModGenAI/Makeitalive/src/SVD/make_dataset_svd.py", line 64, in is_scene_cut
    diff = np.mean(np.abs(frame_a.astype(float) - frame_b.astype(float)))
                                                  ^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
Clips extracted: 8clips [01:03,  7.94s/clips, motion=23.4, rejected_static=0]


KeyboardInterrupt: 

In [ ]:
# ── 6. Vérifier le dataset ──────────────────────────────────────────────────
import os
from PIL import Image
import matplotlib.pyplot as plt

clips = sorted([d for d in os.listdir(DATA_DIR) if d.startswith("clip_")])
print(f"Total clips : {len(clips)}")

if clips:
    preview_clip = clips[min(14, len(clips)-1)]
    frames = sorted(os.listdir(DATA_DIR / preview_clip))
    fig, axes = plt.subplots(1, min(7, len(frames)), figsize=(14, 2))
    if len(frames) == 1:
        axes = [axes]
    for ax, fname in zip(axes, frames[:7]):
        ax.imshow(Image.open(DATA_DIR / preview_clip / fname))
        ax.axis("off")
    plt.suptitle(f"Preview : {preview_clip}", fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 7. Entraînement (run de base : 2000 clips, 3 epochs, LoRA rank 16) ──────
# Temps estimé : ~1.5–2.5h sur A100 / RTX 4090 ; plus long sur GPU moins puissant
# Surveille les lignes "New best saved" → le modèle apprend.

import subprocess, sys

cmd = [
    sys.executable, str(TRAIN_PY),
    "--data_dir",         str(DATA_DIR),
    "--output_dir",       str(CKPT_DIR),
    "--clip_len",         "14",
    "--size",             "512",
    "--epochs",           "3",
    "--batch_size",       "1",
    "--lr",               "1e-4",
    "--lora_rank",        "16",
    "--fps_id",           "7",
    "--motion_bucket_id", "127",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("\n✅ Entraînement terminé.")


In [ ]:
# ── 8. Préparer l'image de test pour l'inférence ────────────────────────────
import shutil, os
from pathlib import Path

# Option A : utiliser la première frame d'un clip existant
clips = sorted([d for d in os.listdir(DATA_DIR) if d.startswith("clip_")])
if clips:
    src_clip  = clips[min(14, len(clips)-1)]
    src_frame = DATA_DIR / src_clip / "frame_000.jpg"
    dest      = Path("test_landscape.jpg")
    shutil.copy(src_frame, dest)
    print(f"✅ Copié depuis {src_frame} → {dest.resolve()}")

# Option B : utiliser ta propre image (décommenter)
# shutil.copy("/chemin/vers/ton/image.jpg", "test_landscape.jpg")


In [ ]:
# ── 9. Inférence avec le meilleur checkpoint ────────────────────────────────
import glob, subprocess, sys
from pathlib import Path

matches = glob.glob(str(CKPT_DIR / "run_*" / "lora_best"))
if not matches:
    print("❌ Aucun checkpoint trouvé. Lance d'abord l'entraînement (cellule 7).")
else:
    lora_best = matches[0]
    print(f"LoRA utilisé : {lora_best}")

    cmd = [
        sys.executable, str(TRAIN_PY),
        "--infer",
        "--lora_dir",   lora_best,
        "--image_path", "test_landscape.jpg",
    ]

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    print("\n✅ Vidéo générée : output.mp4")


In [ ]:
# ── 10. Afficher la vidéo générée ───────────────────────────────────────────
from pathlib import Path
from IPython.display import Video, display

video_path = Path("output.mp4")
if video_path.exists():
    display(Video(str(video_path), embed=True, width=640))
    print(f"Fichier : {video_path.resolve()}")
else:
    print("❌ output.mp4 introuvable — vérifie la cellule d'inférence ci-dessus.")


## Scaling up : 5000 clips, LoRA rank 32

In [ ]:
# ── 11. Chemins pour le run avancé ──────────────────────────────────────────
DATA_DIR_V2 = BASE_DIR / "data_v3"
CKPT_DIR_V2 = BASE_DIR / "checkpoints3"

DATA_DIR_V2.mkdir(parents=True, exist_ok=True)
CKPT_DIR_V2.mkdir(parents=True, exist_ok=True)
print(f"Données   → {DATA_DIR_V2.resolve()}")
print(f"Checkpoints → {CKPT_DIR_V2.resolve()}")


In [ ]:
# ── 12. Extraire le dataset étendu ──────────────────────────────────────────
import subprocess, sys

YOUTUBE_URL = "https://www.youtube.com/watch?v=AKeUssuu3Is"

cmd = [
    sys.executable, str(MAKE_DATASET_PY),
    "--url",        YOUTUBE_URL,
    "--out",        str(DATA_DIR_V2),
    "--frame_gap",  "12",
    "--interval",   "7.0",
    "--max_clips",  "5000",
    "--motion_min", "3.0",
    "--motion_max", "40.0",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("\n✅ Dataset étendu extrait.")


In [ ]:
# ── 13. Entraînement avancé (rank 32, 14 epochs) ────────────────────────────
import subprocess, sys

cmd = [
    sys.executable, str(TRAIN_PY),
    "--data_dir",         str(DATA_DIR_V2),
    "--output_dir",       str(CKPT_DIR_V2),
    "--clip_len",         "14",
    "--size",             "512",
    "--epochs",           "14",
    "--batch_size",       "1",
    "--lr",               "1e-4",
    "--lora_rank",        "32",
    "--fps_id",           "7",
    "--motion_bucket_id", "127",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("\n✅ Entraînement avancé terminé.")


In [ ]:
# ── 14. Inférence avec le run avancé ────────────────────────────────────────
import glob, subprocess, sys

matches = glob.glob(str(CKPT_DIR_V2 / "run_*" / "lora_best"))
if not matches:
    print("❌ Aucun checkpoint — lance d'abord la cellule 13.")
else:
    lora_best = matches[0]
    print(f"LoRA utilisé : {lora_best}")

    cmd = [
        sys.executable, str(TRAIN_PY),
        "--infer",
        "--lora_dir",   lora_best,
        "--image_path", "test_landscape.jpg",
    ]
    subprocess.run(cmd, check=True)
    print("\n✅ output.mp4 généré.")


In [ ]:
# ── 15. Afficher la vidéo finale ────────────────────────────────────────────
from IPython.display import Video, display
from pathlib import Path

p = Path("output.mp4")
if p.exists():
    display(Video(str(p), embed=True, width=640))
else:
    print("❌ output.mp4 introuvable.")
